In [17]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

# Paths
DATASET_PATH = r"C:\Users\hp\OneDrive\Desktop\CESI_UL Internship\Dataset"
OUTPUT_PATH  = r"C:\Users\hp\StressDetection\geometric_features55-56.csv"

# Video settings
FPS         = 35
WINDOW_SIZE = 140  # 4 seconds at 35fps
STEP_SIZE   = 35   # 1 second at 35fps

# MediaPipe setup
mp_face_mesh = mp.solutions.face_mesh

print("Setup complete!")

Setup complete!


In [18]:
def get_landmarks(face_landmarks, w, h):
    """Extract landmark coordinates from MediaPipe output"""
    lm = face_landmarks.landmark
    def pt(idx):
        return np.array([lm[idx].x * w, lm[idx].y * h])
    return lm, pt

def compute_EAR(pt):
    """
    Eye Aspect Ratio — measures how open the eye is
    Formula: EAR = (A + B) / (2 × C)
    A = vertical distance 1
    B = vertical distance 2
    C = horizontal distance
    Low EAR = eye tension = stress indicator
    Normal range: 0.25 to 0.40
    """
    A = np.linalg.norm(pt(159) - pt(145))
    B = np.linalg.norm(pt(158) - pt(153))
    C = np.linalg.norm(pt(33)  - pt(133))
    return (A + B) / (2.0 * C)

def compute_gaze(pt):
    """
    Gaze position — mean x position of iris landmarks
    High std across window = gaze instability = stress indicator
    Landmarks 474-477 = left iris
    """
    iris = np.array([pt(474), pt(475), pt(476), pt(477)])
    return np.mean(iris[:, 0])

def compute_brow(pt, h):
    """
    Eyebrow distance — brow to eye distance normalized by face height
    Low value = furrowed brows = stress indicator
    """
    brow     = pt(70)
    eye      = pt(159)
    face_top = pt(10)
    face_bot = pt(152)
    face_h   = np.linalg.norm(face_top - face_bot)
    brow_eye = np.linalg.norm(brow - eye)
    return brow_eye / face_h if face_h > 0 else 0

def compute_lip(pt):
    """
    Lip activity ratio — vertical opening / horizontal width
    High tension = stress indicator
    Note: inflated during T2 speech task — acknowledged as limitation
    """
    vertical   = np.linalg.norm(pt(13) - pt(14))
    horizontal = np.linalg.norm(pt(61) - pt(291))
    return vertical / horizontal if horizontal > 0 else 0

print("Feature functions defined!")

Feature functions defined!


In [19]:
def apply_sliding_window(features_dict, window_size=140, step_size=35):
    """
    Apply sliding window to extract statistics per window
    
    Window size: 140 frames = 4 seconds at 35fps
    Step size:   35 frames  = 1 second at 35fps
    Overlap:     75%
    
    For each window compute 5 statistics per feature:
    mean, std, min, max, range
    
    Returns list of window dictionaries
    """
    windows  = []
    n_frames = len(features_dict['EAR'])
    start    = 0
    window_id = 0
    
    while start + window_size <= n_frames:
        end = start + window_size
        row = {'window_id': window_id}
        
        for feat_name in ['EAR', 'blink', 'gaze', 'brow', 'lip']:
            values = np.array(features_dict[feat_name][start:end])
            row[f'{feat_name}_mean']  = np.mean(values)
            row[f'{feat_name}_std']   = np.std(values)
            row[f'{feat_name}_min']   = np.min(values)
            row[f'{feat_name}_max']   = np.max(values)
            row[f'{feat_name}_range'] = np.max(values) - np.min(values)
        
        windows.append(row)
        start     += step_size
        window_id += 1
    
    return windows

print("Sliding window function defined!")

Sliding window function defined!


In [20]:
def process_video(video_path):
    """
    Process one video frame by frame
    
    For each frame:
    - Run MediaPipe FaceMesh to get 468 landmarks
    - Compute EAR, gaze, brow, lip
    - Detect blink (EAR < 0.2 for 3+ consecutive frames)
    - If face not detected: carry forward last known values
    
    Track face detection rate for quality control
    
    Returns: (list of window rows, face detection rate)
    """
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        print(f"ERROR: Cannot open {video_path}")
        return [], 0.0
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    features = {'EAR': [], 'blink': [], 'gaze': [], 'brow': [], 'lip': []}
    
    blink_frames     = 0
    last_EAR         = 0.3
    last_gaze        = 0.0
    last_brow        = 0.1
    last_lip         = 0.1
    frames_with_face = 0
    
    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as face_mesh:
        
        for frame_idx in tqdm(range(total_frames), desc="Processing frames"):
            ret, frame = cap.read()
            if not ret:
                break
            
            rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb)
            
            if results.multi_face_landmarks:
                frames_with_face += 1
                lm, pt = get_landmarks(results.multi_face_landmarks[0], w, h)
                
                ear  = compute_EAR(pt)
                gaze = compute_gaze(pt)
                brow = compute_brow(pt, h)
                lip  = compute_lip(pt)
                
                if ear < 0.2:
                    blink_frames += 1
                else:
                    blink_frames = 0
                
                blink     = 1 if blink_frames >= 3 else 0
                last_EAR  = ear
                last_gaze = gaze
                last_brow = brow
                last_lip  = lip
                
            else:
                # Face not detected — carry forward last known values
                ear   = last_EAR
                gaze  = last_gaze
                brow  = last_brow
                lip   = last_lip
                blink = 0
            
            features['EAR'].append(ear)
            features['blink'].append(blink)
            features['gaze'].append(gaze)
            features['brow'].append(brow)
            features['lip'].append(lip)
    
    cap.release()
    
    face_detection_rate = frames_with_face / total_frames
    print(f"Face detection rate: {face_detection_rate:.2%}")
    
    windows = apply_sliding_window(features)
    return windows, face_detection_rate

print("Video processing function defined!")

Video processing function defined!


In [21]:
def apply_baseline_normalization(df):
    feature_cols = [col for col in df.columns 
                    if col not in ['participant', 'task', 'window_id', 'label']]
    
    normalized_df = df.copy()
    normalized_df[feature_cols] = normalized_df[feature_cols].astype(float)
    
    for participant_id in df['participant'].unique():
        t1_mask     = (normalized_df['participant'] == participant_id) & \
                      (normalized_df['task'] == 'T1')
        t1_baseline = normalized_df[t1_mask][feature_cols].mean()
        
        for task in ['T2', 'T3']:
            task_mask = (normalized_df['participant'] == participant_id) & \
                        (normalized_df['task'] == task)
            normalized_df.loc[task_mask, feature_cols] = \
                normalized_df.loc[task_mask, feature_cols] - t1_baseline
    
    return normalized_df

print("Baseline normalization function defined!")

Baseline normalization function defined!


In [22]:
def process_participant(participant_id):
    """
    Process T1, T2, T3 videos for one participant
    
    Tasks:
    T1 = rest task    → label 0 (relaxed)
    T2 = speech task  → label 1 (stressed — social stress)
    T3 = arithmetic   → label 1 (stressed — cognitive stress)
    
    Returns: (list of all window rows, average face detection rate)
    """
    all_rows       = []
    detection_rates = {}
    
    tasks = {'T1': 0, 'T2': 1, 'T3': 1}
    
    for task, label in tasks.items():
        video_name = f"vid_s{participant_id}_{task}.avi"
        video_path = os.path.join(
            DATASET_PATH,
            f"Participant{participant_id}",
            video_name
        )
        
        if not os.path.exists(video_path):
            print(f"WARNING: {video_path} not found — skipping")
            continue
        
        print(f"\nProcessing Participant {participant_id} - {task}...")
        result         = process_video(video_path)
        windows        = result[0]
        detection_rate = result[1]
        detection_rates[task] = detection_rate
        
        for row in windows:
            row['participant'] = participant_id
            row['task']        = task
            row['label']       = label
        
        all_rows.extend(windows)
    
    avg_rate = np.mean(list(detection_rates.values()))
    print(f"\nParticipant {participant_id} average face detection rate: {avg_rate:.2%}")
    
    return all_rows, avg_rate

print("Participant processing function defined!")

Participant processing function defined!


In [23]:
def run_full_pipeline(participant_ids):
    """
    Full pipeline — processes participants and saves to CSV
    
    Strategy:
    - Check if CSV exists → append (never overwrite)
    - Save after each participant → safe to delete videos after
    - Flag participants with low face detection rate (< 50%)
    
    Usage:
    run_full_pipeline([1, 2, 3])  → process participants 1, 2, 3
    run_full_pipeline([4])        → process participant 4 only
    """
    detection_report = {}
    
    for participant_id in participant_ids:
        print(f"\n{'='*50}")
        print(f"PARTICIPANT {participant_id}")
        print(f"{'='*50}")
        
        result   = process_participant(participant_id)
        rows     = result[0]
        avg_rate = result[1]
        detection_report[participant_id] = avg_rate
        
        if len(rows) == 0:
            print(f"WARNING: No data for participant {participant_id} — skipping")
            continue
        
        if avg_rate < 0.5:
            print(f"WARNING: Low face detection rate {avg_rate:.2%} — flagging participant {participant_id}")
        
        df_participant = pd.DataFrame([row for row in rows if isinstance(row, dict)])
        df_participant = apply_baseline_normalization(df_participant)
        
        cols = ['participant', 'task', 'window_id', 'label'] + \
               [c for c in df_participant.columns 
                if c not in ['participant', 'task', 'window_id', 'label']]
        df_participant = df_participant[cols]
        
        if os.path.exists(OUTPUT_PATH):
            df_participant.to_csv(OUTPUT_PATH, index=False, mode='a', header=False)
        else:
            df_participant.to_csv(OUTPUT_PATH, index=False, mode='w')
        
        print(f"Participant {participant_id} saved — {len(df_participant)} rows")
    
    print(f"\n{'='*50}")
    print("FACE DETECTION REPORT:")
    for pid, rate in detection_report.items():
        status = "✓" if rate >= 0.5 else "⚠ LOW"
        print(f"  Participant {pid}: {rate:.2%} {status}")
    
    print(f"\nALL PARTICIPANTS DONE!")
    print(f"CSV saved to: {OUTPUT_PATH}")

print("Full pipeline defined!")

Full pipeline defined!


In [24]:
run_full_pipeline([55, 56])


PARTICIPANT 55

Processing Participant 55 - T1...


Processing frames: 100%|██████████████████████████████████████████████████████████▉| 6325/6326 [03:50<00:00, 27.49it/s]


Face detection rate: 99.98%

Processing Participant 55 - T2...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [04:57<00:00, 21.27it/s]


Face detection rate: 100.00%

Processing Participant 55 - T3...


...

Face detection rate: 98.29%

Processing Participant 56 - T2...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [03:57<00:00, 26.68it/s]


Face detection rate: 100.00%

Processing Participant 56 - T3...


Processing frames: 100%|███████████████████████████████████████████████████████████| 6325/6325 [04:00<00:00, 26.31it/s]


Face detection rate: 98.42%

Participant 56 average face detection rate: 98.90%
Participant 56 saved — 531 rows

FACE DETECTION REPORT:
  Participant 55: 99.99% ✓
  Participant 56: 98.90% ✓

ALL PARTICIPANTS DONE!
CSV saved to: C:\Users\hp\StressDetection\geometric_features55-56.csv
